In [ ]:
from pathlib import Path
from typing import List, Dict
import uuid, numpy as np, torch
import time
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

from open_clip import create_model_from_pretrained
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from qdrant_client.http.exceptions import UnexpectedResponse


: 

In [ ]:
MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
COLLECTION_NAME = "derm_vkb"

def init_biomedclip(device: str | None = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, preprocess = create_model_from_pretrained(MODEL_ID)  # ← matches the doc
    model.to(device).eval()
    return model, preprocess, device


In [ ]:
@torch.no_grad()
def embed_paths_biomedclip(
    img_paths: List[str], model, preprocess, device: str, bs: int = 32
) -> tuple[np.ndarray, List[str]]:
    feats_all: List[np.ndarray] = []
    kept_paths: List[str] = []

    for i in tqdm(range(0, len(img_paths), bs), desc="Embedding (BiomedCLIP)"):
        batch = img_paths[i:i+bs]
        imgs, ok_idx = [], []
        for j, p in enumerate(batch):
            try:
                im = Image.open(p).convert("RGB")
                imgs.append(preprocess(im))              # preprocess from create_model_from_pretrained
                ok_idx.append(j)
            except (FileNotFoundError, UnidentifiedImageError):
                continue
        if not imgs:
            continue

        images = torch.stack(imgs).to(device)           # float32 tensors
        # For image-only embeddings, use encode_image (cleaner than forward(images, texts))
        img_feats = model.encode_image(images)          # (B, D)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        feats_all.append(img_feats.cpu().numpy().astype("float32"))

        for j in ok_idx:
            kept_paths.append(batch[j])

        del images, img_feats
        if device.startswith("cuda"):
            torch.cuda.empty_cache()

    if not feats_all:
        return np.zeros((0, 512), dtype=np.float32), []
    return np.vstack(feats_all), kept_paths

In [ ]:
def build_qdrant_index(
    train_items: List[Dict],
    collection: str = COLLECTION_NAME,
    qdrant_url: str = "http://127.0.0.1:6333",
    bs: int = 32,
    batch_upsert: int = 512,
) -> int:
    # 1) init model (per doc)
    model, preprocess, device = init_biomedclip()

    # 2) filter readable images & resolve paths
    items_ok, paths = [], []
    for it in train_items:
        p = str(Path(it["image_path"]).resolve())
        try:
            Image.open(p).close()
            items_ok.append(it)
            paths.append(p)
        except Exception:
            continue
    if not paths:
        print("No valid images found.")
        return 0

    # 3) embed
    vecs, kept_paths = embed_paths_biomedclip(paths, model, preprocess, device, bs=bs)
    # align kept items to kept paths
    path2item = {str(Path(it["image_path"]).resolve()): it for it in items_ok}
    kept_items = [path2item[p] for p in kept_paths]
    assert len(kept_items) == vecs.shape[0], "Mismatch between embeddings and items."

    client = QdrantClient(url=qdrant_url, grpc_port=6334, timeout=1200.0, prefer_grpc=True)

    # 5) collection
    dim = int(vecs.shape[1])
    client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
        timeout=1200,
    )

    # 6) upsert
    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=v.tolist(),
            payload={
                "image_path": p,
                "age": it.get("age"),
                "sex": it.get("sex"),
                "diagnosis": it.get("diagnosis"),
                "anatom_site": it.get("anatom_site"),
                "melanocytic": it.get("melanocytic"),
            },
        )
        for v, it, p in zip(vecs, kept_items, kept_paths)
    ]

    n = 0
    for i in tqdm(range(0, len(points), batch_upsert), desc="Upserting"):
        client.upsert(collection_name=collection, points=points[i:i+batch_upsert])
        n += len(points[i:i+batch_upsert])

    print(f"Indexed {n} points into '{collection}' (dim={dim}).")
    return n

In [ ]:
import pickle as pkl
TRAIN_DATATASET_PATH = "../dataset/isic_train_set.pkl"

with open(TRAIN_DATATASET_PATH, "rb") as f:
    train_set = pkl.load(f)

: 

In [ ]:
image_path = "../dataset/train"
for item in train_set:
    file_name = Path(item["image_path"]).name
    item["image_path"] = f"{image_path}/{file_name}"

In [ ]:
print(train_set[0]["image_path"])

In [ ]:
build_qdrant_index(train_set, collection=COLLECTION_NAME, qdrant_url="http://localhost:6333")